# 07 — Fenotipi CROSS-COHORT: Track#3 vs coorte ORIGINALE (tesi)
Domanda: i (deboli) fenotipi di Track#3 puntano nella **stessa direzione cerebrale** di quelli
della coorte originale (74 sogg, C0 fronto-motor / C1 fronto-occipital)?

Metodo (da **V5W_09**): entrambe le coorti → covarianze SPD → media di Riemann → **canali condivisi**
(56, inclusi **F3** e **PO8**, gli hub del fenotipo) → clustering orientato per PO8 → confronto:
1. **correlazione spaziale** della difference-map C1−C0 (node strength) sui 56 canali,
2. **coseno degli assi fenotipici** in uno spazio tangente COMUNE,
3. **test diretto F3–PO8**: è più alto in C1 in entrambe le coorti?

> ⚠️ **Caveat forte**: il fenotipo Track#3 è silhouette ~0.03 (quasi rumore, n=15). Anche una
> correlazione positiva va presa come *esplorativa*. Ma questo è il test corretto.
> Richiede il **raw originale sul server** (`data/raw_csv/training_set`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr
from pyriemann.tangentspace import TangentSpace
import track3_config as C, track3_io as io, track3_phenotypes as PH
print(C.summary()); assert C.DATA_ROOT is not None, C._no_data_msg()

## §1 — Covarianze delle due coorti + canali condivisi

In [ ]:
# coorte originale (61ch, sul server; cache dopo il 1° run ~minuti)
try:
    M_OG, subj_og, clab_og = PH.load_original_cohort_covariances()
    print('OG:', M_OG.shape, '|', len(subj_og), 'soggetti')
except FileNotFoundError as e:
    print('⚠️', e); raise
# Track#3 (64ch)
data = PH.load_covariances(); M_T3, clab_t3 = data['M'], data['clab']
shared = PH.channel_intersection(clab_t3, clab_og)
print(f'canali condivisi: {len(shared)}  (F3={"F3" in shared}, PO8={"PO8" in shared})')
M_OGr = PH.restrict_covariances(M_OG, clab_og, shared)
M_T3r = PH.restrict_covariances(M_T3, clab_t3, shared)
print('ristrette:', M_OGr.shape, M_T3r.shape)

## §2 — Clustering Riemann di ciascuna coorte (orientato per PO8)
C1 = cluster con node-strength più alto su PO8 (rende confrontabili le label tra coorti).

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
def cluster_and_orient(M):
    lab, TS, Z = PH.riemann_cluster(M, k=2)
    lab = PH.orient_by_electrode(M, lab, shared, anchor='PO8')
    Zs = PCA(min(20,len(M)-1), random_state=42).fit_transform(StandardScaler().fit_transform(TS))
    return lab, silhouette_score(Zs, lab)
lab_og, sil_og = cluster_and_orient(M_OGr)
lab_t3, sil_t3 = cluster_and_orient(M_T3r)
print(f'OG   : cluster {np.bincount(lab_og)}  silhouette={sil_og:.3f}')
print(f'Track3: cluster {np.bincount(lab_t3)}  silhouette={sil_t3:.3f}')
print('(OG dovrebbe essere forte ~0.3+, Track3 debole ~0.03 => vedi caveat)')
# concordanza OG re-cluster vs label canoniche della tesi (EEG_16b), se allineabili
try:
    from sklearn.metrics import adjusted_rand_score
    canon = PH.original_phenotype_labels()
    al = np.array([canon.get(s, -1) for s in subj_og])
    ok = al >= 0
    if ok.sum() > 5:
        print(f'ARI(OG re-cluster, EEG_16b canoniche) = {adjusted_rand_score(al[ok], np.array(lab_og)[ok]):.3f}')
except Exception as _e:
    print('(label canoniche non allineate:', _e, ')')

## §3 — TEST 1: correlazione spaziale delle difference-map C1−C0
Per ogni canale, forza di connettività C1−C0. Se i due vettori (OG e Track#3) correlano positivamente,
le due coorti hanno la **stessa firma spaziale** del fenotipo.

In [ ]:
def ns_diff(M, lab):
    return M[lab==1].mean(0).mean(1) - M[lab==0].mean(0).mean(1)
d_og, d_t3 = ns_diff(M_OGr, lab_og), ns_diff(M_T3r, lab_t3)
r, p = pearsonr(d_og, d_t3)
print(f'Correlazione spaziale difference-map (56 canali): r={r:+.3f}  p={p:.4f}')
print('  r>0 e p<0.05 => stessa firma spaziale; r~0 => fenotipi diversi/indipendenti')
fig, ax = plt.subplots(1,2, figsize=(12,4.5))
ax[0].scatter(d_og, d_t3, s=40); ax[0].axhline(0,color='k',lw=.6); ax[0].axvline(0,color='k',lw=.6)
for i,c in enumerate(shared):
    if abs(d_og[i])>np.percentile(abs(d_og),80) or abs(d_t3[i])>np.percentile(abs(d_t3),80):
        ax[0].annotate(c,(d_og[i],d_t3[i]),fontsize=7)
ax[0].set_xlabel('OG  C1−C0'); ax[0].set_ylabel('Track3  C1−C0'); ax[0].set_title(f'Firma spaziale: r={r:+.3f} p={p:.3f}')
ax[1].barh(range(len(shared)), d_og, color='#2166AC', alpha=.6, label='OG')
ax[1].barh(range(len(shared)), d_t3, color='#D6604D', alpha=.6, label='Track3')
ax[1].set_yticks(range(len(shared))); ax[1].set_yticklabels(shared, fontsize=5); ax[1].legend(); ax[1].set_title('C1−C0 per canale')
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_crosscohort_spatial.png', dpi=130); plt.show()

## §4 — TEST 2: coseno degli assi fenotipici in spazio tangente COMUNE
Metto OG+Track#3 nello STESSO tangent space (media di Riemann congiunta, det=1) e misuro il coseno
tra l'asse C1−C0 di OG e quello di Track#3. Coseno ~1 = stesso asse; ~0 = ortogonali.

In [ ]:
allM = np.concatenate([M_OGr, M_T3r], 0)
allMn = PH.det1_normalize(allM)
TSc = TangentSpace().fit(allMn).transform(allMn)
nog = len(M_OGr)
ax_og = TSc[:nog][lab_og==1].mean(0) - TSc[:nog][lab_og==0].mean(0)
ax_t3 = TSc[nog:][lab_t3==1].mean(0) - TSc[nog:][lab_t3==0].mean(0)
cos = float(ax_og @ ax_t3 / (np.linalg.norm(ax_og)*np.linalg.norm(ax_t3) + 1e-12))
print(f'Coseno assi fenotipici (tangent comune): {cos:+.3f}')
print('  ~+1 stesso asse | ~0 ortogonali | ~-1 opposti')

## §5 — TEST 3: diretto su F3–PO8 (l'edge-firma della tesi)
Nella tesi il fenotipo C1 è definito da F3→PO8 alto. È più alto in C1 in ENTRAMBE le coorti?

In [ ]:
i_f3, i_po8 = shared.index('F3'), shared.index('PO8')
def edge_c1_c0(M, lab):
    return M[lab==1][:, i_f3, i_po8].mean(), M[lab==0][:, i_f3, i_po8].mean()
og1, og0 = edge_c1_c0(M_OGr, lab_og); t31, t30 = edge_c1_c0(M_T3r, lab_t3)
print(f'F3-PO8  OG    : C1={og1:.3g}  C0={og0:.3g}  ->  {"C1>C0 ✓" if og1>og0 else "C1<C0 ✗"}')
print(f'F3-PO8  Track3: C1={t31:.3g}  C0={t30:.3g}  ->  {"C1>C0 ✓" if t31>t30 else "C1<C0 ✗"}')
print('  concordi (F3-PO8 più alto in C1 in entrambe) =>', (og1>og0)==(t31>t30))

## §6 — Topomap affiancate C1−C0 (OG vs Track#3)

In [ ]:
import mne
info = mne.create_info(shared, C.FS, ch_types='eeg')
clab_pos, pos = io.canonical_positions(); posmap={PH._canon(c):p for c,p in zip(clab_pos,pos)}
info.set_montage(mne.channels.make_dig_montage(ch_pos={c:posmap[c] for c in shared if c in posmap}, coord_frame='head'), on_missing='warn')
fig, ax = plt.subplots(1,2, figsize=(9,4))
for a,(t,dvec) in zip(ax, [('OG C1−C0', d_og), ('Track3 C1−C0', d_t3)]):
    mne.viz.plot_topomap(dvec, info, axes=a, show=False, cmap='RdBu_r'); a.set_title(t)
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_crosscohort_topomap.png', dpi=130); plt.show()

## Conclusioni
Interpreta insieme i 3 test:
- **§3 correlazione spaziale** r: se positiva e significativa → stessa firma; se ~0 → fenotipi diversi.
- **§4 coseno assi**: conferma indipendente in geometria comune.
- **§5 F3–PO8**: test diretto sull'edge-firma della tesi.

⚠️ **Ricorda il caveat**: il fenotipo Track#3 è silhouette ~0.03. Se i 3 test concordano (r>0, cos>0,
F3-PO8 concorde) è un segnale *esplorativo* interessante che l'asse fronto-occipitale esiste anche
qui, seppur debole. Se sono nulli/discordi → i fenotipi NON replicano su Track#3 (probabile, n=15).
In entrambi i casi è una risposta onesta e verificata, non pattern-matching su rumore.